# 05 - Wartime official channels

A method showcase, not a claim about the war. The 2026 US/Israel-Iran war ran
inside the collection window, so it is an opportunity to point the instrument at
a live event and see what the layers pick up.

**The sampling problem, and why this notebook exists.** An earlier pass compared
Farsi-majority channels found by the snowball against a Hebrew-majority
comparison group, and reported an apparent Iranian volume surge. That result was
a sampling artefact: the snowball reaches channels *because they are being
forwarded*, so a cohort defined by language is enriched for whatever is
circulating at the time. It is retracted, and the notebooks that produced it are
not part of this repository.

The fix is to fix the cohort. Two hand-seeded lists of **official** channels,
state ministries, armed forces and state news agencies, one per side, crawled
into their own databases with the same scraper and the same settings. The cohort
is chosen before the war rather than discovered during it, so a change in volume
is a change in behaviour and not a change in who is in the sample.

**Figures produced here**

| Output | Thesis figure |
|---|---|
| `official_volume_index` | wartime volume against each cohort's own pre-war level |
| `per_channel_lift` | pooled vs per-channel volume lift |
| `institution_arms` | one institution, several language arms |
| `homefront_clock` | Israeli Home Front alerts against the Iranian state wire |
| `framing_convergence` | daily cross-side narrative overlap |
| `toxicity_by_side` | comment-group audience toxicity |

**Running this.** The two companion databases, `tgdataset_israel` and
`tgdataset_iran`, were dropped after the analysis and are not shipped, so the
database-backed sections here cannot be re-run even with a corpus in hand. The
SQL is carried because it is the record of what was computed; the shipped
summaries in `../data/` are what the thesis quotes, and the sections marked
*offline* render from them.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

from _shared import DATA, FIGURES, ISRAEL_DB, IRAN_DB, csv, q, setup_plots, shape

setup_plots()

## 0. Windows, events and cohorts

The comparison is a pre-war baseline against the war window, each cohort measured
against its own baseline rather than against the other side. That is what makes
the two sides comparable at all: they have very different absolute volumes.

In [ ]:
ANALYSIS_SINCE = "2025-06-01"
PRE = ("2025-11-01", "2026-02-28")   # 3-month pre-war baseline, [lo, hi)
WAR = ("2026-02-28", "2026-04-30")   # war and immediate aftermath, [lo, hi)
PRE_DAYS = (pd.Timestamp(PRE[1]) - pd.Timestamp(PRE[0])).days
WAR_DAYS = (pd.Timestamp(WAR[1]) - pd.Timestamp(WAR[0])).days

EVENTS = [("2026-02-28", "war begins"),
          ("2026-04-08", "ceasefire"),
          ("2026-04-13", "naval blockade")]
EVENTS_DT = [(pd.Timestamp(d), lbl) for d, lbl in EVENTS]

# Local-clock offsets for the tempo read. Iran is UTC+3:30 year-round; Israel runs
# UTC+2 in winter and UTC+3 from late March. The war window straddles the Israeli
# change, so the Israeli peak shifts about an hour later in April: a caveat for the
# clock figure, not a confound for the night/office-hours shares.
IRAN_UTC_OFFSET = "3 hours 30 minutes"
ISRAEL_UTC_OFFSET = "2 hours"

print("pre:", PRE, f"({PRE_DAYS}d) | war:", WAR, f"({WAR_DAYS}d)")
display(csv("cohorts.csv"))

In [ ]:
SQL_SEED_COHORT = """
    SELECT c.channel_id,
           (SELECT username FROM metadata_snapshots s WHERE s.channel_id = c.channel_id
             ORDER BY captured_at DESC LIMIT 1) AS username,
           (SELECT title FROM metadata_snapshots s WHERE s.channel_id = c.channel_id
             ORDER BY captured_at DESC LIMIT 1) AS title,
           (SELECT count(*) FROM messages m WHERE m.channel_id = c.channel_id) AS msgs
    FROM channels c
    WHERE c.source IN ('dataset', 'seed_file')
"""
# Cohort language comes from the seed list, not from language detection: the
# non-domestic arms are enumerated by hand and everything else is the domestic
# language (Iran -> fa, Israel -> he). That keeps the language axis independent of
# any model, which matters because the whole point is an institution-fixed read.

IRAN_EN = {1006487902,    # Press TV
           1710170626,    # IRNA English
           1361842824,    # Tehran Times
           2281366764,    # Mehr English
           1002489027,    # Khamenei English
           1253933606}    # Tasnim English
ISRAEL_EN = {1341028511,  # IDF English
             1802807592,  # The Jerusalem Post
             1607417124,  # The Times of Israel
             1224147708}  # Israel MFA
ISRAEL_AR = {1672983921}  # IDF Spokesperson, Arabic

# Institution-fixed parallel-language channels: one institution, two or more
# languages, so a divergence between them is audience targeting and cannot be
# channel selection.
IDF  = {"IDF \u00b7 Hebrew":   (1155294424, ISRAEL_DB),
        "IDF \u00b7 English":  (1341028511, ISRAEL_DB),
        "IDF \u00b7 Arabic":   (1672983921, ISRAEL_DB)}
IRNA = {"IRNA \u00b7 Persian": (1068487996, IRAN_DB),
        "IRNA \u00b7 English": (1710170626, IRAN_DB)}
MEHR = {"Mehr \u00b7 Persian": (1002765601, IRAN_DB),
        "Mehr \u00b7 English": (2281366764, IRAN_DB)}
HOMEFRONT = 1441886157   # Home Front Command missile-alert channel

## 1. Volume against each cohort's own baseline (*offline*)

Every series is indexed to 100 at its own pre-war mean, so the two sides can share
an axis. The two retracted snowball cohorts are plotted faded alongside, which is
the point of the figure: they barely move, while the fixed official cohorts do.
That contrast is the retraction made visible.

In [ ]:
SQL_DAILY = """
    SELECT (date AT TIME ZONE 'UTC')::date AS d, count(*) AS msgs
    FROM messages
    WHERE channel_id = ANY(:ids) AND date >= CAST(:s AS timestamptz)
    GROUP BY 1 ORDER BY 1
"""

SQL_MODAL_REF = """
    WITH t AS (SELECT channel_id, lang, count(*) n FROM message_languages GROUP BY 1, 2),
    tot AS (SELECT channel_id, sum(n) d FROM t GROUP BY 1),
    modal AS (SELECT DISTINCT ON (t.channel_id) t.channel_id, t.lang,
                     t.n::numeric / tot.d AS share, tot.d AS dets
              FROM t JOIN tot USING (channel_id) ORDER BY t.channel_id, t.n DESC)
    SELECT channel_id FROM modal WHERE lang = :lang AND share >= 0.5 AND dets >= 20
"""
# indexed(): reindex to a full daily range, 7-day centred mean, divide by the
# pre-war mean and scale to 100. Per-window lift is
# (war messages / WAR_DAYS) / (pre messages / PRE_DAYS).

vol = csv("volume_lift.csv")
display(vol)
print("The two snowball reference rows are the retracted nb14/nb15 cohorts, kept "
      "in the table so the artefact is visible next to the fixed cohorts.")

### The volume curve

The daily series behind the curve is one row per cohort per day across two
databases that no longer exist, so this section needs those databases. The lift
table above is what the thesis quotes.

In [ ]:
def daily(ids, db=None):
    if not ids:
        return pd.DataFrame(columns=["d", "msgs"])
    df = q(SQL_DAILY, db=db, ids=ids, s=ANALYSIS_SINCE)
    df["d"] = pd.to_datetime(df["d"])
    return df

def indexed(df):
    s = (df.set_index("d")["msgs"]
           .reindex(pd.date_range(df["d"].min(), df["d"].max()), fill_value=0)
           .rolling(7, center=True, min_periods=1).mean())
    pre = s[(s.index >= PRE[0]) & (s.index < PRE[1])].mean()
    return s / pre * 100 if pre else s

il = q(SQL_SEED_COHORT, db=ISRAEL_DB)
ir = q(SQL_SEED_COHORT, db=IRAN_DB)
IL_ALL = il.loc[il["msgs"] > 0, "channel_id"].astype(int).tolist()
IR_ALL = ir.loc[ir["msgs"] > 0, "channel_id"].astype(int).tolist()
fa_ref = q(SQL_MODAL_REF, lang="fa")["channel_id"].astype(int).tolist()
he_ref = q(SQL_MODAL_REF, lang="he")["channel_id"].astype(int).tolist()

LINES = [("Israel", IL_ALL, ISRAEL_DB, "#3b7dd8", 1.00, "-"),
         ("Iran",   IR_ALL, IRAN_DB,   "#d4584f", 1.00, "-"),
         ("Iran snowball-fa (retracted ref)",   fa_ref, None, "#d4584f", 0.35, "--"),
         ("Israel snowball-he (retracted ref)", he_ref, None, "#3b7dd8", 0.35, "--")]

fig, ax = plt.subplots(figsize=(5.48, 2.32))
for name, ids, db, color, alpha, ls in LINES:
    df = daily(ids, db)
    if df.empty:
        continue
    ax.plot(indexed(df), ls, color=color, lw=1.0, alpha=alpha, label=f"{name} (n={len(ids)})")
ax.axhline(100, color="#888", lw=0.7, ls=":")
ax.set_yscale("log")
ax.set_ylabel("msgs/day 7d-MA  (100 = own pre-war mean)")
ax.set_title("Wartime volume vs each cohort's own pre-war level (snowball baselines faded)")
_, ymax = ax.get_ylim()
for dt, lbl in EVENTS_DT:
    if lbl == "ceasefire":   # omitted here, kept in the other panels
        continue
    ax.axvline(dt, color="#444", lw=0.8, ls=":")
    ax.text(dt, ymax * 0.92, lbl, rotation=90, va="top", ha="right", fontsize=7, color="#444")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b'%y"))
ax.legend(loc="lower left")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES / "official_volume_index.png", bbox_inches="tight")
fig.savefig(FIGURES / "official_volume_index.pdf", bbox_inches="tight")
plt.show()

## 2. Pooled lift is not the typical channel's lift

A pooled, volume-weighted lift is dominated by the largest channels. The
per-channel distribution answers a different question: did most of the cohort
escalate, or a few?

The box shows per-channel lift (clipped for display) and the diamond marks the
pooled figure, so the gap between them is legible. Channels with a negligible
pre-war rate are excluded, since their lift ratio is unstable by construction.

The per-channel lifts need the companion databases; the summary reads offline.

In [ ]:
SQL_PER_CHANNEL_LIFT = """
    SELECT channel_id,
           count(*) FILTER (WHERE date >= CAST(:p0 AS timestamptz)
                             AND date <  CAST(:p1 AS timestamptz)) AS pre,
           count(*) FILTER (WHERE date >= CAST(:w0 AS timestamptz)
                             AND date <  CAST(:w1 AS timestamptz)) AS war
    FROM messages
    WHERE channel_id = ANY(:ids) AND date >= CAST(:p0 AS timestamptz)
    GROUP BY 1
"""
# Rates are per day (pre / PRE_DAYS, war / WAR_DAYS); channels below 0.05 msgs/day
# pre-war are dropped, then lift = war rate / pre rate.

pcl = csv("per_channel_lift.csv", index_col=0)
display(pcl)

def per_channel_lift(ids, db=None):
    df = q(SQL_PER_CHANNEL_LIFT, db=db, ids=ids,
           p0=PRE[0], p1=PRE[1], w0=WAR[0], w1=WAR[1])
    df["pre_r"] = df["pre"] / PRE_DAYS
    df["war_r"] = df["war"] / WAR_DAYS
    act = df[df["pre_r"] > 0.05].copy()
    act["lift"] = act["war_r"] / act["pre_r"]
    return act

SIDES = [("Israel", IL_ALL, ISRAEL_DB, "#3b7dd8"),
         ("Iran",   IR_ALL, IRAN_DB,   "#d4584f")]
pcl_box = {name: per_channel_lift(ids, db)["lift"].clip(upper=20)
           for name, ids, db, _ in SIDES}

fig, ax = plt.subplots(figsize=(5.48, 3.9))
order = [n for n, *_ in SIDES if len(pcl_box.get(n, []))]
colors = {n: c for n, _, _, c in SIDES}
bp = ax.boxplot([pcl_box[n].values for n in order], showfliers=False, widths=0.5,
                patch_artist=True, medianprops=dict(color="#222", lw=0.9))
for patch, n in zip(bp["boxes"], order):
    patch.set_facecolor(colors[n])
    patch.set_alpha(0.45)
for i, n in enumerate(order, start=1):
    ax.scatter([i], [pcl.loc[n, "pooled_lift"]], marker="D", s=45, color=colors[n],
               edgecolor="black", zorder=5,
               label="pooled (vol-weighted)" if i == 1 else None)
ax.axhline(1.0, color="#444", ls="--", lw=0.8)
# Linear y from 0, so the typical lift near 1x reads directly; headroom for labels.
_wmax = max((float(np.nanmax(line.get_ydata())) for line in bp["whiskers"]), default=2.0)
_top = max([_wmax] + [float(pcl.loc[n, "pooled_lift"]) for n in order]) * 1.35
ax.set_ylim(0, _top)
for i, n in enumerate(order, start=1):
    ax.annotate(f"{pcl.loc[n, 'pct_silent']:.0f}% silent\n{pcl.loc[n, 'pct_up']:.0f}% up",
                (i, _top * 0.96), ha="center", va="top", fontsize=7.5, color="#555")
ax.set_xticks(range(1, len(order) + 1))
ax.set_xticklabels([f"{n}\n(n={int(pcl.loc[n, 'n_active'])})" for n in order], fontsize=7.5)
ax.set_ylabel("per-channel volume lift (war / pre-war)", fontsize=7.5)
ax.set_title("Pooled (diamond) vs per-channel box: Israel vs Iran", fontsize=8.5)
ax.legend(loc="lower right", fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES / "per_channel_lift.png", bbox_inches="tight")
fig.savefig(FIGURES / "per_channel_lift.pdf", bbox_inches="tight")
plt.show()

## 3. One institution, several audiences (*offline*)

The cleanest comparison in the notebook. Each institution here runs parallel
channels in different languages: the same organisation, the same editorial
authority, different target audiences. A divergence between the arms cannot be
channel selection, because the channel is held fixed at the institution.

Each bar is that arm's wartime lift against its **own** pre-war level, so a small
channel that tripled and a large one that tripled read the same.

In [ ]:
arms_df = csv("institution_arms.csv")
display(arms_df)

LANG_COLOR = {"Hebrew": "#3b7dd8", "Persian": "#d4584f",
              "English": "#e09a3c", "Arabic": "#2e9e6b"}

def _lang_key(arm):
    for k in LANG_COLOR:
        if k in arm:
            return k
    return "English"

fig, ax = plt.subplots(figsize=(5.48, 3.4))
ypos, ylabels = [], []
y = 0
for inst in ["IDF (Israel)", "IRNA (Iran)", "Mehr (Iran)"]:
    for _, r in arms_df[arms_df["institution"] == inst].iterrows():
        ypos.append(y)
        ylabels.append(r["arm"])
        ax.barh(y, r["lift"], color=LANG_COLOR[_lang_key(r["arm"])], alpha=0.85)
        ax.text(r["lift"], y, f"  {r['lift']:.1f}x", va="center", fontsize=7)
        y += 1
    y += 0.6
# Tehran Times is English-only, with no domestic sibling: the standout
# international arm, shown apart from the paired institutions.
r = arms_df[arms_df["arm"].str.startswith("Tehran Times")].iloc[0]
ypos.append(y)
ylabels.append("Tehran Times (en-only)")
ax.barh(y, r["lift"], color=LANG_COLOR["English"], alpha=0.85)
ax.text(r["lift"], y, f"  {r['lift']:.1f}x", va="center", fontsize=8.5)

ax.axvline(1.0, color="#444", ls="--", lw=0.8)
ax.set_xlim(0, ax.get_xlim()[1] * 1.08)   # headroom so the last label is not clipped
ax.set_yticks(ypos)
ax.set_yticklabels(ylabels, fontsize=7.5)
ax.invert_yaxis()
ax.set_xlabel("wartime volume lift (war / pre-war, vs the arm's own pre-war level)",
              fontsize=7.5)
ax.set_title("Institution-fixed: each state escalates its English arm hardest", fontsize=8.5)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=k) for k, c in LANG_COLOR.items()],
          fontsize=7, loc="upper right")
fig.tight_layout()
fig.savefig(FIGURES / "institution_arms.png", bbox_inches="tight")
fig.savefig(FIGURES / "institution_arms.pdf", bbox_inches="tight")
plt.show()

## 4. The air-war clock

The Israeli Home Front Command channel publishes one message per missile-threat
alert, which makes it an unusually direct proxy for the physical air war rather
than for anyone's messaging strategy. Plotting it against the Iranian state wire
puts a communications series and a kinetic series on the same axis.

The daily series comes from both companion databases, so this section needs them.
The tempo summary in `diurnal_clock.csv` is the shipped companion: it records how
each cohort's share of night-time and office-hours posting moved between the
pre-war and war windows, on each side's own local clock.

In [ ]:
display(csv("diurnal_clock.csv"))

pikud = daily([HOMEFRONT], ISRAEL_DB)
iro = daily(IR_ALL, IRAN_DB)
span = pd.date_range("2026-01-01", "2026-05-15")
pk = pikud.set_index("d")["msgs"].reindex(span, fill_value=0)
ir_s = iro.set_index("d")["msgs"].reindex(span, fill_value=0)

fig, ax = plt.subplots(figsize=(5.48, 2.11))
ax.bar(pk.index, pk.values, color="#3b7dd8", alpha=0.5, width=1.0,
       label="Home Front Command alerts/day (missile threat)")
ax2 = ax.twinx()
ax2.plot(ir_s.index, ir_s.rolling(3, center=True, min_periods=1).mean(),
         color="#d4584f", lw=1.0, label="Iranian state channels, 3d-MA")
ax.set_ylabel("Home Front alerts/day")
ax2.set_ylabel("Iranian state msgs/day")
ax.set_title("Air-war clock: Israeli Home Front alerts vs Iranian state wire")
for dt, lbl in EVENTS_DT:
    ax.axvline(dt, color="#444", lw=0.8, ls=":")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=7)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES / "homefront_clock.png", bbox_inches="tight")
fig.savefig(FIGURES / "homefront_clock.pdf", bbox_inches="tight")
plt.show()

war_mask = (span >= pd.Timestamp(WAR[0])) & (span < pd.Timestamp(WAR[1]))
r = float(np.corrcoef(pk[war_mask], ir_s[war_mask])[0, 1])
print(f"Home Front alerts in the war window: {int(pk[war_mask].sum()):,}  |  "
      f"corr(Home Front, official-Iran volume): r = {r:.3f}")

## 5. Do the two sides talk about the same thing? (*offline*)

Each day's Hebrew broadcast sample and Farsi broadcast sample are embedded with a
multilingual sentence encoder, averaged into a centroid per side, and the cosine
between the two centroids is the day's narrative overlap. High means both sides
are discussing the same events; low means they are running separate agendas.

Sampling is capped per side per window and ordered by a content hash, so the draw
is deterministic and neither side's high-volume days dominate. Days with fewer
than 20 messages on either side are dropped.

The embedding step is a frozen input: the per-day cosines are shipped and this
section replots them, so no encoder is needed to read the result. The caveat from
the thesis applies: a cross-lingual embedding gives direction and contrast, not a
calibrated similarity scale.

In [ ]:
SQL_BROADCAST_SAMPLE = """
    SELECT (date AT TIME ZONE 'UTC')::date AS d, text
    FROM messages
    WHERE channel_id = ANY(:ids)
      AND date >= CAST(:lo AS timestamptz) AND date < CAST(:hi AS timestamptz)
      AND text IS NOT NULL AND char_length(text) >= :ml
    ORDER BY md5(text || message_id::text) LIMIT :cap
"""
CAP, MINLEN = 9000, 25
EMBEDDER = "paraphrase-multilingual-MiniLM-L12-v2"
# Per day: centroid of each side's embeddings, then
#   cos = (c_il @ c_ir) / (norm(c_il) * norm(c_ir))
# keeping only days with at least 20 sampled messages on each side.

conv = csv("framing_convergence.csv", parse_dates=["d"])

fig, ax = plt.subplots(figsize=(5.48, 1.94))
ax.plot(conv["d"], conv["cos"].rolling(3, center=True, min_periods=1).mean(),
        "-o", ms=2, color="#7d3bd8")
ax.set_ylabel("cross-side centroid cosine\n(daily narrative overlap)")
ax.set_title("Cross-lingual narrative convergence: Israeli (he) vs Iranian (fa), by day")
for dt, lbl in EVENTS_DT:
    ax.axvline(dt, color="#444", lw=0.8, ls=":")
    ax.text(dt, ax.get_ylim()[1], lbl, rotation=90, va="top", ha="right",
            fontsize=8, color="#444")
ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES / "framing_convergence.png", bbox_inches="tight")
fig.savefig(FIGURES / "framing_convergence.pdf", bbox_inches="tight")
plt.show()
print(f"convergence: {len(conv)} days \u00b7 mean {conv['cos'].mean():.3f} "
      f"\u00b7 peak {conv['cos'].max():.3f} on {conv.loc[conv['cos'].idxmax(), 'd'].date()}")

## 6. Audience toxicity (*offline*)

The layer above measured what the states published. This one measures what their
readers wrote back, in the comment groups attached to the official channels.

Both sides are scored with the same classifier on samples drawn the same way, so
the comparison is like for like at the sampling level. It is not like for like at
the model level, and the thesis says so: Arabic and English are in the
classifier's training data, while Hebrew and Persian are zero-shot transfer. The
firmest cross-side reading is therefore the by-language panel at a shared
language, not the headline by-side bar.

The scores are a frozen input; the per-message text is not shipped.

In [ ]:
SQL_AUDIENCE_SAMPLE = """
    SELECT m.text FROM messages m JOIN channels c USING (channel_id)
    WHERE c.source = 'linked_group'
      AND m.date >= CAST(:lo AS timestamptz) AND m.date < CAST(:hi AS timestamptz)
      AND m.text IS NOT NULL AND char_length(m.text) >= :ml
    ORDER BY md5(m.text || m.message_id::text) LIMIT :cap
"""
TOX_MODEL = "textdetox/xlmr-large-toxicity-classifier"
TOX_THR, MINLEN_AUD = 0.5, 20
# Language per message from langdetect (seeded), toxicity from the classifier
# above; pct_toxic is the share at or above TOX_THR.

by_side = csv("toxicity_by_side.csv", index_col=0)
by_lang = csv("toxicity_by_lang.csv")
by_lang = by_lang[by_lang["n"] >= 50]

fig, axes = plt.subplots(2, 1, figsize=(5.48, 3.4), gridspec_kw={"height_ratios": [3, 1.3]})
pv = by_lang.pivot(index="lang", columns="side", values="pct_toxic")
pv.plot(kind="bar", ax=axes[0],
        color={"Israel audience": "#3b7dd8", "Iran audience": "#d4584f"})
axes[0].set_ylabel(f"% toxic (p>={TOX_THR})")
axes[0].set_title("Audience toxicity by language (n>=50)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

ypos = np.arange(len(by_side)) * 0.4          # pack the thin bars close together
axes[1].barh(ypos, by_side["pct_toxic"], height=0.27, color=["#d4584f", "#3b7dd8"])
axes[1].set_yticks(ypos)
axes[1].set_yticklabels(by_side.index)
axes[1].set_ylim(-0.27, ypos[-1] + 0.27)      # snug box around the two bars
axes[1].set_xlabel(f"% messages toxic (p>={TOX_THR})")
axes[1].set_title("Audience toxicity by side")
fig.suptitle("Comment-group audience toxicity in the war window", y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(FIGURES / "toxicity_by_side.png", bbox_inches="tight")
fig.savefig(FIGURES / "toxicity_by_side.pdf", bbox_inches="tight")
plt.show()
display(by_side)
display(by_lang)

## Reading

Both official cohorts escalate against their own pre-war baselines, and both do so
far more than the language-defined snowball cohorts that the earlier pass
mistook for the same signal. That is the methodological result: the cohort
definition, not the war, produced the earlier finding.

Within an institution, the international arm escalates hardest. The IDF's English
channel outruns its Hebrew one, and the Iranian agencies' English wires outrun
their Persian ones. Because the institution is fixed, that is a statement about
audience targeting: wartime messaging is aimed outward more than inward.

The pooled and per-channel views disagree in the useful way. Pooled lift is
carried by a handful of large channels, while the median channel moves far less,
so a single headline multiplier would misdescribe the cohort.

Every number here is a within-cohort rate over hand-picked official channels
during one window. It describes what these institutions did, and nothing about
Telegram at large.